# R50 — find where cassette proposals are lost

This finishes the interrupted R49 experiment using the **already trained PR137 candidate**. It performs inference only, on the same 20 original development photographs. It compares native hard NMS with one fixed linear Soft-NMS rule using the **same decoded tensor**. The frozen666 primary, historical663, R38 and R50 proposal references are scored separately.

**Publication status:** source and script pins are fixed to the published implementation below.

In Google Colab select **Runtime → Change runtime type → T4 GPU**, then **Run all**. Return `baomai-r50-diagnosis-results.zip`. CPU execution also works, more slowly. The two CPU preflight cells run before installation or inference. Existing completed returns are verified and reused; incompatible checkouts or outputs are preserved and require a fresh base directory.


In [ ]:
# 1. Restore the exact source and repair sparse membership on every setup run.
import hashlib, json, os, re, subprocess, sys
from pathlib import Path
SOURCE_REVISION = '5aa5c04d91fb6e1b5a5cf9cce85215f7564238aa'
SCRIPT_SHA256 = 'cba4f1df6dcf431c933af1506a4de15718636d9477406eb34779ac4d8ab6ee3e'
R50_REFERENCE_SHA256 = 'f639459bdd986f6553b9e00f0ecfdf1eee3c7f5344e95c300a4a17a66352c04f'
assert re.fullmatch(r'[0-9a-f]{40}', SOURCE_REVISION), 'Publication incomplete: source revision is not pinned.'
assert re.fullmatch(r'[0-9a-f]{64}', SCRIPT_SHA256), 'Publication incomplete: script SHA256 is not pinned.'
BASE = Path(os.environ.get('BAOMAI_R50_BASE', '/content/baomai-r50-' + SOURCE_REVISION[:12]))
SOURCE, WORK = BASE / 'source', BASE / 'run'
REPOSITORY = os.environ.get('BAOMAI_R50_REPOSITORY', 'https://github.com/Siuuuers/dl.git')
ENV = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1', GIT_TERMINAL_PROMPT='0', PYTHONDONTWRITEBYTECODE='1')
def git(*args, capture=False, text_input=None):
    result = subprocess.run(['git', '-C', str(SOURCE), *args], check=True, env=ENV,
                            input=text_input, text=True, capture_output=capture)
    return result.stdout.strip() if capture else None
FRESH = not (SOURCE / '.git').exists()
if not FRESH:
    assert git('rev-parse', 'HEAD', capture=True) == SOURCE_REVISION, 'Wrong revision preserved. Set BAOMAI_R50_BASE to a fresh directory.'
    assert not git('status', '--porcelain', '--untracked-files=all', capture=True), 'Modified checkout preserved. Set BAOMAI_R50_BASE to a fresh directory.'
else:
    assert not SOURCE.exists() or not any(SOURCE.iterdir()), 'Existing source files preserved. Choose a fresh BAOMAI_R50_BASE.'
    SOURCE.mkdir(parents=True, exist_ok=True)
    git('init', '-q')
    git('remote', 'add', 'origin', REPOSITORY)
    git('fetch', '--depth=1', '--filter=blob:none', 'origin', SOURCE_REVISION)
patterns = ['/scripts/', '/data/manifest.json', '/data/labels/',
            '/data/candidates/front-review-2026-11-26-r37/',
            '/data/reviews/additional-fronts-2026-11-27-r47/',
            '/data/reviews/fronts-2026-11-26-r38/', '/data/reviews/fronts-2026-11-27-r48/',
            '/data/reviews/fronts-2026-11-27-r50/',
            '/data/references/sequence-refined-reference-2026-11-18/',
            '/data/references/visible-front-completion-2026-11-14/',
            '/results/review/front-review-20261126-r38/row-maps.json']
git('sparse-checkout', 'init', '--no-cone')
git('sparse-checkout', 'set', '--no-cone', '--stdin', text_input='\n'.join(patterns) + '\n')
if FRESH: git('checkout', '--detach', SOURCE_REVISION)
SCRIPT = SOURCE / 'scripts/diagnose_fronts_r50.py'
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == SCRIPT_SHA256, 'Source script hash differs.'
COMMON = ['--root', str(SOURCE), '--work', str(WORK), '--source-revision', SOURCE_REVISION,
          '--script-sha256', SCRIPT_SHA256, '--r50-reference-sha256', R50_REFERENCE_SHA256]
subprocess.run([sys.executable, '-B', str(SCRIPT), *COMMON, '--source-check-only'], check=True, env=ENV)


## Verify all inputs on CPU

This restores the exact PR137 return, replays its saved scores, verifies the candidate checkpoint and checks the SHA256 and original 1280 × 720 dimensions of all 20 PNGs. It also binds every scored reference label and the R38 stable row map. No package installation, model loading or fresh inference occurs here. Optional environment paths support an already downloaded archive, originals or cache.


In [ ]:
# 2. CPU input preflight. A rerun must match the existing immutable input contract.
INPUT_OPTIONS = []
for variable, flag in [('BAOMAI_R50_ARCHIVE', '--archive'), ('BAOMAI_R50_ORIGINALS', '--originals-dir'), ('BAOMAI_R50_CACHE', '--cache-dir')]:
    if os.environ.get(variable): INPUT_OPTIONS += [flag, os.environ[variable]]
COMMON += INPUT_OPTIONS
subprocess.run([sys.executable, '-B', str(SCRIPT), *COMMON, '--preflight-only'], check=True, env=ENV)
PREFLIGHT = json.loads((WORK / 'preflight.json').read_text())
print('Verified input contract:', PREFLIGHT['contract_sha256'])
print('Score references:', {name: value['objects'] for name, value in PREFLIGHT['contract']['references'].items() if name != 'r48_parent'})
RESULT = WORK / 'baomai-r50-diagnosis-results.zip'
COMPLETED = RESULT.exists()  # Existing returns were fully checked by the preflight command.


## Run the fixed comparison

Both arms use confidence **> 0.13**, overlap threshold **0.70**, at most **300** detections, and the same postclip filter. Soft-NMS only decays a proposal's score by `1 − IoU` when its overlap exceeds 0.70. Every proposal keeps its own two keypoints. The native-index reconstruction must exactly match the fresh hard-NMS prediction; otherwise the run stops.

Miss traces distinguish absent IoU-eligible geometry, the confidence gate, ordinary NMS/cap, postclip suppression, and retained-but-unmatched assignment. These are pipeline locations, not proven physical causes. Saved and fresh baseline differences are reported. The four reference versions use the same scenes and are not independent test sets. Nothing is automatically promoted.


In [ ]:
# 3. Install the pinned inference package only after both CPU preflights pass.
import importlib.metadata as metadata
DEVICE = 'cpu'
if not COMPLETED:
    import torch
    DEVICE = '0' if torch.cuda.is_available() else 'cpu'
    constraints = WORK / 'runtime-constraints.txt'
    constraints.write_text('\n'.join(name + '==' + metadata.version(name) for name in ('torch', 'torchvision')) + '\n')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--constraint', str(constraints),
                    'ultralytics==8.4.182'], check=True, env=ENV)
    assert metadata.version('ultralytics') == '8.4.182'
    subprocess.run([sys.executable, '-B', str(SCRIPT), '--selftest'], check=True, env=ENV)
    print('Inference device:', DEVICE, '| PyTorch:', torch.__version__)
else:
    print('Completed return verified: skipping package installation and inference.')


In [ ]:
# 4. Run, or verify/reuse the completed return; then download the audit bundle.
import zipfile
mode = ['--verify-return'] if COMPLETED else ['--device', DEVICE]
subprocess.run([sys.executable, '-B', str(SCRIPT), *COMMON, *mode], check=True, env=ENV)
with zipfile.ZipFile(RESULT) as archive:
    summary = json.loads(archive.read('summary.json'))
for name, reports in summary['reports'].items():
    print('\nReference:', name)
    for arm in ('saved', 'hard', 'soft'):
        d = reports[arm]['detection']
        print(arm, {key: d[key] for key in ('tp', 'fn', 'fp', 'duplicate_fp', 'other_fp')})
print('Fresh miss mechanisms:', summary['fresh_miss_mechanisms'])
print('Return:', RESULT, '| SHA256:', hashlib.sha256(RESULT.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(RESULT))
except ImportError:
    print('Download the ZIP from the path above.')
